# 00 · Regresión lineal simple: la geometría de los mínimos cuadrados

**Módulo 3 · Sesión 6** — Regresión lineal

## Objetivos

`01-regresion-lineal.md` define la regresión por mínimos cuadrados con fórmulas. Este
notebook las **dibuja**, con una sola variable y pocos puntos para que todo se vea:

1. El SSE como lo que su nombre dice: una **suma de áreas de cuadrados**, uno por residual.
2. Por qué la recta de mínimos cuadrados es la que deja el área total más pequeña.
3. El $R^2$ como una **comparación de dos áreas**: los cuadrados alrededor de la media
   frente a los cuadrados alrededor de la recta, y la descomposición
   $\text{SS}_{\text{tot}} = \text{SS}_{\text{reg}} + \text{SS}_{\text{res}}$.
4. La función de pérdida $\mathcal{L}(\beta_0, \beta_1)$ como un **paraboloide**: cada recta
   es un punto de esa superficie y la de mínimos cuadrados es su fondo.
5. 🔵 Opcional: la misma historia vista desde $\mathbb{R}^n$, donde el ajuste es una
   **proyección ortogonal** y el $R^2$ es un coseno al cuadrado.

Es el punto de partida del módulo: el notebook 01 desciende por esta misma superficie,
ahora con seis predictores.

**Paquetes:** `numpy`, `pandas`, `matplotlib`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-3-regresion-evaluacion/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Pocos datos, para poder verlos

Usamos el mismo `rendimiento-estudiantes.csv` del módulo 1, con la regresión simple que
allí se ajustó por descenso del gradiente: `promedio_anterior` → `nota_final`. Tomamos solo
**10 estudiantes** al azar: con 400 los cuadrados de la sección 3 serían una mancha.

Hay una ventaja geométrica en esta elección: las dos variables están en la **misma escala**
(notas de 0 a 5). Con `aspect="equal"` un residual de 0.5 mide lo mismo en vertical que en
horizontal, y su cuadrado se ve como un cuadrado de verdad.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
muestra = datos.sample(n=10, random_state=SEMILLA).sort_values("promedio_anterior")

x = muestra["promedio_anterior"].to_numpy()
y = muestra["nota_final"].to_numpy()
n = len(x)

muestra[["id_estudiante", "promedio_anterior", "nota_final"]].reset_index(drop=True)

In [ ]:
LIM_X = (2.4, 5.0)
LIM_Y = (1.9, 5.0)


def preparar_eje(eje, titulo=""):
    eje.set_xlim(*LIM_X)
    eje.set_ylim(*LIM_Y)
    eje.set_aspect("equal")
    eje.set_xlabel("promedio_anterior")
    eje.set_ylabel("nota_final")
    eje.set_title(titulo)


fig, eje = plt.subplots(figsize=(5.5, 5.5))
eje.scatter(x, y, color="black", zorder=3)
preparar_eje(eje, "10 estudiantes")
plt.show()

## 2. Una recta cualquiera y sus residuales

Una recta es un par de números $(\beta_0, \beta_1)$. Para cada estudiante predice
$\hat{y}_i = \beta_0 + \beta_1 x_i$, y el **residual** es lo que la recta no explica:

$$
e_i = y_i - \hat{y}_i
$$

Geométricamente, $e_i$ es la **distancia vertical** del punto a la recta — vertical, no
perpendicular, porque lo que se quiere predecir es $y$ y el error se mide en unidades de
$y$. Empecemos con una recta trazada "a ojo".

In [ ]:
def predecir(b0, b1, x):
    return b0 + b1 * x


def residuales(b0, b1, x, y):
    return y - predecir(b0, b1, x)


def sse(b0, b1, x, y):
    return np.sum(residuales(b0, b1, x, y) ** 2)


def dibujar_recta(eje, b0, b1, **kwargs):
    xs = np.array(LIM_X)
    eje.plot(xs, predecir(b0, b1, xs), **kwargs)


def dibujar_residuales(eje, b0, b1, x, y, color="tab:red"):
    for xi, yi, yhat in zip(x, y, predecir(b0, b1, x)):
        eje.plot([xi, xi], [yi, yhat], color=color, linewidth=1.5, zorder=2)


A_OJO = (0.5, 0.8)  # (beta_0, beta_1) trazada a ojo

fig, eje = plt.subplots(figsize=(5.5, 5.5))
dibujar_recta(eje, *A_OJO, color="tab:blue", label="recta a ojo")
dibujar_residuales(eje, *A_OJO, x, y)
eje.scatter(x, y, color="black", zorder=3)
eje.legend(loc="upper left")
preparar_eje(eje, rf"$\hat y = {A_OJO[0]} + {A_OJO[1]}\,x$")
plt.show()

pd.DataFrame(
    {
        "x": x,
        "y": y,
        "y_hat": predecir(*A_OJO, x).round(3),
        "residual": residuales(*A_OJO, x, y).round(3),
    }
)

## 3. El SSE es un área

Para resumir los diez residuales en un solo número no basta con sumarlos: los positivos y
los negativos se cancelarían. Mínimos cuadrados los **eleva al cuadrado**:

$$
\text{SSE}(\beta_0, \beta_1) = \sum_{i=1}^{n} e_i^2 = \sum_{i=1}^{n}
\bigl(y_i - \beta_0 - \beta_1 x_i\bigr)^2
$$

Y $e_i^2$ tiene una lectura literal: es el **área de un cuadrado de lado $|e_i|$**. Si
sobre cada segmento rojo de la figura anterior construimos su cuadrado, el SSE es el área
total pintada. Ajustar la regresión es mover la recta hasta que esa área sea mínima.

In [ ]:
def dibujar_cuadrados(eje, b0, b1, x, y, color="tab:red", alpha=0.25):
    """Un cuadrado de lado |e_i| apoyado sobre cada residual, hacia la derecha."""
    for xi, yi, yhat in zip(x, y, predecir(b0, b1, x)):
        lado = abs(yi - yhat)
        eje.add_patch(
            Rectangle((xi, min(yi, yhat)), lado, lado, facecolor=color, edgecolor=color,
                      alpha=alpha, zorder=1)
        )


def panel_cuadrados(eje, b0, b1, titulo, color_recta="tab:blue"):
    dibujar_cuadrados(eje, b0, b1, x, y)
    dibujar_residuales(eje, b0, b1, x, y)
    dibujar_recta(eje, b0, b1, color=color_recta)
    eje.scatter(x, y, color="black", zorder=3)
    preparar_eje(eje, f"{titulo}\nSSE = {sse(b0, b1, x, y):.3f}")


fig, eje = plt.subplots(figsize=(5.5, 5.5))
panel_cuadrados(eje, *A_OJO, "Recta a ojo")
plt.show()

Los cuadrados se superponen en algunos puntos; no importa, cada uno cuenta por separado.
Lo que importa es la **desproporción**: un residual el doble de grande pinta un cuadrado
**cuatro** veces más grande. Por eso mínimos cuadrados es tan sensible a los puntos
alejados: un solo estudiante mal predicho puede dominar el área total.

Comparemos tres rectas: la trazada a ojo, la horizontal en la media $\bar y$ (la referencia
trivial que ignora $x$) y la de **mínimos cuadrados**, que para una sola variable tiene
fórmula cerrada:

$$
\hat\beta_1 = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2}
\qquad\qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x
$$

Es el caso $p = 1$ de la ecuación normal $\boldsymbol{\beta} =
(\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}$; la sección 5 muestra de
dónde sale.

In [ ]:
x_media, y_media = x.mean(), y.mean()
b1_ols = np.sum((x - x_media) * (y - y_media)) / np.sum((x - x_media) ** 2)
b0_ols = y_media - b1_ols * x_media
OLS = (b0_ols, b1_ols)
MEDIA = (y_media, 0.0)

# Comprobación contra NumPy (polyfit devuelve [pendiente, intercepto])
assert np.allclose(np.polyfit(x, y, deg=1), [b1_ols, b0_ols])
print(f"Mínimos cuadrados: beta_0 = {b0_ols:.4f}, beta_1 = {b1_ols:.4f}")

fig, ejes = plt.subplots(1, 3, figsize=(15, 5.5))
panel_cuadrados(ejes[0], *A_OJO, "Recta a ojo")
panel_cuadrados(ejes[1], *MEDIA, r"Recta horizontal en $\bar y$", color_recta="tab:green")
panel_cuadrados(ejes[2], *OLS, "Mínimos cuadrados", color_recta="tab:purple")
plt.tight_layout()
plt.show()

La recta de mínimos cuadrados deja el área total más pequeña de las tres — y de **todas**
las rectas posibles, como se verá en la sección 5. Y hay una sorpresa: la recta a ojo,
que a la vista sigue la nube razonablemente, pinta **más** área que la horizontal, que
ignora $x$ por completo. Su pendiente es demasiado fuerte: en los dos extremos se separa
de los datos —por debajo a la izquierda, por encima a la derecha— y paga allí cuadrados
que la horizontal no paga. El ojo no suma áreas bien.

Dos propiedades de la recta de mínimos cuadrados que se leen en la figura y se comprueban
en una línea:

- Pasa por el **punto de medias** $(\bar x, \bar y)$: es lo que dice la fórmula de
  $\hat\beta_0$.
- Sus residuales **suman cero**: los segmentos rojos hacia arriba compensan exactamente
  a los de hacia abajo. La recta a ojo no cumple ninguna de las dos.

In [ ]:
e_ols = residuales(*OLS, x, y)
e_ojo = residuales(*A_OJO, x, y)
print(f"Recta OLS en x̄:  {predecir(*OLS, x_media):.4f}   vs  ȳ = {y_media:.4f}")
print(f"Suma de residuales OLS:    {e_ols.sum():+.2e}")
print(f"Suma de residuales a ojo:  {e_ojo.sum():+.4f}")

## 4. El $R^2$ es una razón entre dos áreas

Un SSE de 1.5 ¿es mucho o poco? Depende de cuánto varían las notas de por sí. El $R^2$
responde comparando contra la recta horizontal en $\bar y$, la que no usa $x$ para nada —
su SSE de la sección anterior, 1.977, es justamente la variabilidad total de las notas.
Hay tres áreas en juego, cada una con sus propios cuadrados:

| Área | Cuadrado de lado | Qué mide |
|---|---|---|
| $\text{SS}_{\text{tot}} = \sum_i (y_i - \bar y)^2$ | distancia del punto a la **media** | Variabilidad total de $y$, sin modelo |
| $\text{SS}_{\text{reg}} = \sum_i (\hat y_i - \bar y)^2$ | distancia de la **recta** a la media | Lo que la recta explica |
| $\text{SS}_{\text{res}} = \sum_i (y_i - \hat y_i)^2$ | distancia del punto a la **recta** | Lo que queda sin explicar (el SSE) |

In [ ]:
y_hat = predecir(*OLS, x)
ss_tot = np.sum((y - y_media) ** 2)
ss_reg = np.sum((y_hat - y_media) ** 2)
ss_res = np.sum((y - y_hat) ** 2)


def cuadrados_entre(eje, x, desde, hasta, color):
    for xi, a, b in zip(x, desde, hasta):
        lado = abs(a - b)
        eje.plot([xi, xi], [a, b], color=color, linewidth=1.5, zorder=2)
        eje.add_patch(
            Rectangle((xi, min(a, b)), lado, lado, facecolor=color, edgecolor=color,
                      alpha=0.25, zorder=1)
        )


fig, ejes = plt.subplots(1, 3, figsize=(15, 5.5))
paneles = [
    (y, np.full(n, y_media), "tab:green", rf"$SS_{{tot}}$ = {ss_tot:.3f}"),
    (y_hat, np.full(n, y_media), "tab:orange", rf"$SS_{{reg}}$ = {ss_reg:.3f}"),
    (y, y_hat, "tab:red", rf"$SS_{{res}}$ = {ss_res:.3f}"),
]
for eje, (desde, hasta, color, titulo) in zip(ejes, paneles):
    eje.axhline(y_media, color="tab:green", linestyle="--", linewidth=1, label=r"$\bar y$")
    dibujar_recta(eje, *OLS, color="tab:purple", label="OLS")
    cuadrados_entre(eje, x, desde, hasta, color)
    eje.scatter(x, y, color="black", zorder=3)
    eje.scatter(x, y_hat, color="tab:purple", marker="x", zorder=3)
    preparar_eje(eje, titulo)
    eje.legend(loc="upper left")
plt.tight_layout()
plt.show()

Para la recta de mínimos cuadrados (con intercepto), las áreas cumplen una identidad
exacta — un teorema de Pitágoras, como se verá en la sección 7:

$$
\text{SS}_{\text{tot}} = \text{SS}_{\text{reg}} + \text{SS}_{\text{res}}
$$

Así que el área verde se reparte entre la naranja (explicada) y la roja (sin explicar), y
el $R^2$ es la **fracción del área verde que la recta logra explicar**:

$$
R^2 = \frac{\text{SS}_{\text{reg}}}{\text{SS}_{\text{tot}}} = 1 -
\frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}
$$

In [ ]:
r2 = 1 - ss_res / ss_tot
r_pearson = np.corrcoef(x, y)[0, 1]

print(f"SS_tot             = {ss_tot:.4f}")
print(f"SS_reg + SS_res    = {ss_reg + ss_res:.4f}")
print(f"R² = 1 - SS_res/SS_tot = {r2:.4f}")
print(f"R² = SS_reg/SS_tot     = {ss_reg / ss_tot:.4f}")
print(f"r de Pearson al cuadrado = {r_pearson**2:.4f}")

fig, eje = plt.subplots(figsize=(8, 2.2))
eje.barh(0, ss_reg, color="tab:orange", label=r"$SS_{reg}$ (explicada)")
eje.barh(0, ss_res, left=ss_reg, color="tab:red", label=r"$SS_{res}$ (sin explicar)")
eje.axvline(ss_tot, color="tab:green", linestyle="--", label=r"$SS_{tot}$")
eje.text(ss_reg / 2, 0, f"R² = {r2:.2f}", ha="center", va="center", color="white",
         fontweight="bold")
eje.set_yticks([])
eje.set_xlabel("Área (suma de cuadrados)")
eje.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
plt.tight_layout()
plt.show()

Con una sola variable, además, $R^2 = r^2$: el cuadrado de la correlación de Pearson entre
$x$ e $y$. Es una coincidencia exclusiva de la regresión simple; con varios predictores el
$R^2$ ya no es el cuadrado de ninguna correlación individual.

### La identidad solo vale en el mínimo

La descomposición **no** es una propiedad de cualquier recta: depende de que los residuales
sean ortogonales a la recta, y eso solo lo garantiza mínimos cuadrados. Con la recta a ojo,
las áreas no cuadran — y $1 - \text{SS}_{\text{res}}/\text{SS}_{\text{tot}}$ puede incluso
salir **negativo**, que es lo que `01-regresion-lineal.md` advierte para el $R^2$ en test:
un modelo peor que predecir siempre la media.

In [ ]:
filas = []
for nombre, (b0, b1) in [("mínimos cuadrados", OLS), ("a ojo", A_OJO), ("pendiente 2.0", (y_media - 2.0 * x_media, 2.0))]:
    y_hat_r = predecir(b0, b1, x)
    res = np.sum((y - y_hat_r) ** 2)
    reg = np.sum((y_hat_r - y_media) ** 2)
    filas.append(
        {
            "recta": nombre,
            "SS_reg": reg,
            "SS_res": res,
            "SS_reg + SS_res": reg + res,
            "SS_tot": ss_tot,
            "1 - SS_res/SS_tot": 1 - res / ss_tot,
        }
    )
pd.DataFrame(filas).round(3)

Las dos rectas que no son de mínimos cuadrados tienen un "$R^2$" **negativo**: su área
roja supera a la verde, como ya se vio para la recta a ojo en la sección 3. La de
pendiente 2.0 incluso pasa por el punto de medias, y no le basta. Para ambas
$\text{SS}_{\text{reg}} + \text{SS}_{\text{res}} \neq \text{SS}_{\text{tot}}$: la
descomposición solo existe en el fondo del paraboloide, que es lo que sigue.

Por eso $R^2$ en entrenamiento, calculado con la recta de mínimos cuadrados, está siempre
entre 0 y 1; en test la recta ya no es la óptima para esos datos y el $R^2$ puede caer
por debajo de cero.

## 5. La pérdida como paraboloide

Hasta ahora cada recta fue una figura en el plano $(x, y)$. Cambiemos de espacio: cada
recta es también un **punto** $(\beta_0, \beta_1)$, y su SSE es una **altura**. Al recorrer
todos los pares posibles se obtiene una superficie, $\mathcal{L}(\beta_0, \beta_1) =
\text{SSE}$. Desarrollando el cuadrado:

$$
\mathcal{L}(\beta_0, \beta_1) = n\beta_0^2 + \Bigl(\sum x_i^2\Bigr)\beta_1^2 + 2\Bigl(\sum
x_i\Bigr)\beta_0\beta_1 - 2\Bigl(\sum y_i\Bigr)\beta_0 - 2\Bigl(\sum x_i y_i\Bigr)\beta_1 +
\sum y_i^2
$$

Un polinomio de grado 2 en $(\beta_0, \beta_1)$ con términos cuadrados positivos: un
**paraboloide** que abre hacia arriba. Tiene un único fondo, y ese fondo es la recta de
mínimos cuadrados.

In [ ]:
B0 = np.linspace(b0_ols - 2, b0_ols + 2, 200)
B1 = np.linspace(b1_ols - 0.6, b1_ols + 0.6, 200)
malla_b0, malla_b1 = np.meshgrid(B0, B1)

# SSE en toda la malla de una vez: (200, 200, 1) frente a (10,) se difunde a (200, 200, 10)
residuales_malla = y - (malla_b0[..., None] + malla_b1[..., None] * x)
malla_sse = np.sum(residuales_malla**2, axis=-1)

rectas = {
    "a ojo": (A_OJO, "tab:blue"),
    "horizontal en ȳ": (MEDIA, "tab:green"),
    "mínimos cuadrados": (OLS, "tab:purple"),
}

fig = plt.figure(figsize=(13, 5.5))
eje3d = fig.add_subplot(1, 2, 1, projection="3d")
eje3d.plot_surface(malla_b0, malla_b1, malla_sse, cmap="viridis", alpha=0.75,
                   rstride=5, cstride=5, linewidth=0)
for nombre, ((b0, b1), color) in rectas.items():
    if B0[0] <= b0 <= B0[-1] and B1[0] <= b1 <= B1[-1]:
        eje3d.scatter(b0, b1, sse(b0, b1, x, y), color=color, s=60, depthshade=False,
                      label=nombre)
eje3d.set_xlabel(r"$\beta_0$")
eje3d.set_ylabel(r"$\beta_1$")
eje3d.set_zlabel("SSE")
eje3d.view_init(elev=28, azim=-60)
eje3d.set_title("Superficie de pérdida")
eje3d.legend(loc="upper left")

eje = fig.add_subplot(1, 2, 2)
niveles = np.quantile(malla_sse, np.linspace(0.002, 0.9, 20))
contorno = eje.contour(malla_b0, malla_b1, malla_sse, levels=niveles, cmap="viridis",
                       linewidths=0.8)
for nombre, ((b0, b1), color) in rectas.items():
    if B0[0] <= b0 <= B0[-1] and B1[0] <= b1 <= B1[-1]:
        eje.plot(b0, b1, "o", color=color, markersize=8, label=nombre)
eje.set_xlabel(r"$\beta_0$")
eje.set_ylabel(r"$\beta_1$")
eje.set_title("Curvas de nivel (vista desde arriba)")
eje.legend()
plt.tight_layout()
plt.show()

print(f"SSE en el fondo: {malla_sse.min():.4f}  (OLS: {ss_res:.4f})")

Dos cosas saltan a la vista:

- El fondo de la superficie coincide con la recta de mínimos cuadrados; la recta a ojo
  está más arriba, en la ladera.
- La superficie no es un cuenco redondo sino un **valle largo y estrecho en diagonal**. A
  lo largo del valle el SSE casi no cambia: subir $\beta_1$ y bajar $\beta_0$ a la vez
  produce rectas que giran alrededor del centro de los datos y se ajustan casi igual de
  bien. Las elipses de nivel están inclinadas porque $\beta_0$ y $\beta_1$ están
  **acoplados**: el término cruzado $2(\sum x_i)\beta_0\beta_1$ es grande, ya que las notas
  rondan 3.6 y no 0.

### Cortes: la parábola de una variable

Si fijamos $\beta_0$ en su valor óptimo y movemos solo la pendiente, cortamos el
paraboloide con un plano vertical. El corte es una **parábola** en $\beta_1$, con el mínimo
exactamente en $\hat\beta_1$ — y cada punto de la curva es una recta distinta del plano
$(x, y)$.

In [ ]:
pendientes = np.linspace(b1_ols - 0.3, b1_ols + 0.3, 200)
corte = [sse(b0_ols, b1, x, y) for b1 in pendientes]
muestras_b1 = [b1_ols - 0.15, b1_ols, b1_ols + 0.15]
colores_muestra = ["tab:blue", "tab:purple", "tab:orange"]

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ejes[0].plot(pendientes, corte, color="gray")
for b1, color in zip(muestras_b1, colores_muestra):
    ejes[0].plot(b1, sse(b0_ols, b1, x, y), "o", color=color, markersize=9)
ejes[0].set_xlabel(r"$\beta_1$  (con $\beta_0 = \hat\beta_0$ fijo)")
ejes[0].set_ylabel("SSE")
ejes[0].set_title("Corte del paraboloide: una parábola")

for b1, color in zip(muestras_b1, colores_muestra):
    dibujar_recta(ejes[1], b0_ols, b1, color=color, label=rf"$\beta_1$ = {b1:.2f}")
ejes[1].scatter(x, y, color="black", zorder=3)
ejes[1].legend(loc="upper left")
preparar_eje(ejes[1], "Las tres rectas correspondientes")
plt.tight_layout()
plt.show()

### Centrar $x$ endereza el valle

El acoplamiento entre $\beta_0$ y $\beta_1$ no es una propiedad de los datos sino de dónde
está el origen. Si usamos $x - \bar x$ en lugar de $x$, el término cruzado desaparece
($\sum (x_i - \bar x) = 0$) y los ejes del paraboloide quedan alineados con $\beta_0$ y
$\beta_1$. La forma de la superficie la decide la matriz $\mathbf{X}^{\top}\mathbf{X}$ —
su hessiana es $2\mathbf{X}^{\top}\mathbf{X}$ — y el cociente entre sus valores propios
(el **número de condición**) mide qué tan alargado es el valle.

In [ ]:
def matriz_disenio(x):
    return np.column_stack([np.ones_like(x), x])


x_c = x - x_media
for nombre, xx in [("x original", x), ("x centrada", x_c)]:
    valores_propios = np.linalg.eigvalsh(matriz_disenio(xx).T @ matriz_disenio(xx))
    print(f"{nombre:>11}: valores propios de XᵀX = {valores_propios.round(3)}, "
          f"número de condición = {valores_propios.max() / valores_propios.min():.0f}")

b0_c = y_media  # con x centrada, el intercepto óptimo es la media de y
C0 = np.linspace(b0_c - 1, b0_c + 1, 200)
C1 = np.linspace(b1_ols - 1, b1_ols + 1, 200)
malla_c0, malla_c1 = np.meshgrid(C0, C1)
malla_sse_c = np.sum((y - (malla_c0[..., None] + malla_c1[..., None] * x_c)) ** 2, axis=-1)

fig = plt.figure(figsize=(13, 5.5))
eje3d = fig.add_subplot(1, 2, 1, projection="3d")
eje3d.plot_surface(malla_c0, malla_c1, malla_sse_c, cmap="viridis", alpha=0.8,
                   rstride=5, cstride=5, linewidth=0)
eje3d.scatter(b0_c, b1_ols, ss_res, color="tab:purple", s=60, depthshade=False)
eje3d.set_xlabel(r"$\beta_0$ (con $x-\bar x$)")
eje3d.set_ylabel(r"$\beta_1$")
eje3d.set_zlabel("SSE")
eje3d.view_init(elev=28, azim=-60)
eje3d.set_title("x centrada: el paraboloide como cuenco")

eje = fig.add_subplot(1, 2, 2)
eje.contour(malla_c0, malla_c1, malla_sse_c, levels=20, cmap="viridis", linewidths=0.8)
eje.plot(b0_c, b1_ols, "o", color="tab:purple", markersize=8, label="mínimos cuadrados")
eje.set_aspect("equal")
eje.set_xlabel(r"$\beta_0$ (con $x-\bar x$)")
eje.set_ylabel(r"$\beta_1$")
eje.set_title("Curvas de nivel: elipses con ejes alineados")
eje.legend()
plt.tight_layout()
plt.show()

print(f"SSE en el fondo, x centrada: {malla_sse_c.min():.4f}  (OLS: {ss_res:.4f})")

Ahora sí se ve el paraboloide como un cuenco: el número de condición pasó de más de mil
a 4, y las elipses tienen sus ejes paralelos a $\beta_0$ y $\beta_1$ (siguen siendo
elipses y no círculos porque $\mathbf{X}^{\top}\mathbf{X}$ centrada es diagonal, pero con
$n=10$ en una entrada y $\sum (x_i - \bar x)^2 \approx 2.35$ en la otra).

Mismos datos, misma pendiente óptima, mismo SSE mínimo: centrar no cambia **qué** recta se
ajusta, solo las coordenadas con que se describe. Pero cambia todo para quien tiene que
**caminar** por esa superficie. El descenso del gradiente avanza perpendicular a las
curvas de nivel; en el valle inclinado zigzaguea de pared a pared, en el centrado baja
casi recto. Es la razón geométrica detrás de la estandarización que el notebook 01 muestra
en código, donde sin escalar el descenso directamente diverge.

## 6. Del fondo del paraboloide a la ecuación normal

En el fondo de un paraboloide el plano tangente es horizontal: las dos derivadas parciales
valen cero.

$$
\frac{\partial \mathcal{L}}{\partial \beta_0} = -2\sum_i e_i = 0 \qquad\qquad
\frac{\partial \mathcal{L}}{\partial \beta_1} = -2\sum_i x_i e_i = 0
$$

Cada ecuación tiene una lectura geométrica que ya vimos:

- La primera dice que los residuales **suman cero** — la propiedad de la sección 3.
- La segunda dice que los residuales **no están correlacionados con $x$**: no queda en
  ellos ninguna tendencia lineal que la recta pudiera aprovechar.

En forma matricial las dos son una sola, $\mathbf{X}^{\top}\mathbf{e} = \mathbf{0}$, es
decir $\mathbf{X}^{\top}(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}) = \mathbf{0}$, y
despejando sale la **ecuación normal** de `01-regresion-lineal.md`:

$$
\mathbf{X}^{\top}\mathbf{X}\,\boldsymbol{\beta} = \mathbf{X}^{\top}\mathbf{y}
$$

In [ ]:
X = matriz_disenio(x)
beta_normal = np.linalg.solve(X.T @ X, X.T @ y)

print(f"Ecuación normal: beta = {beta_normal.round(6)}")
print(f"Fórmula cerrada: beta = {np.array(OLS).round(6)}")
print(f"Xᵀe en el mínimo:      {(X.T @ e_ols).round(12)}")
print(f"Xᵀe con la recta a ojo: {(X.T @ e_ojo).round(4)}")

## 7. 🔵 Opcional: la vista desde $\mathbb{R}^n$

Hay una tercera forma de mirar lo mismo, la más abstracta y la más reveladora. En lugar de
$n$ puntos en el plano, pensemos en **tres vectores de $\mathbb{R}^n$**: $\mathbf{y}$ (las
$n$ notas), $\mathbf{1}$ (un vector de unos) y $\mathbf{x}$. Toda predicción lineal
$\hat{\mathbf{y}} = \beta_0\mathbf{1} + \beta_1\mathbf{x}$ es una combinación de
$\mathbf{1}$ y $\mathbf{x}$, así que vive en el **plano** que ellos generan (el espacio
columna de $\mathbf{X}$). Minimizar $\lVert \mathbf{y} - \hat{\mathbf{y}} \rVert^2$ es
buscar el punto de ese plano más cercano a $\mathbf{y}$: la **proyección ortogonal**.

Con $n = 10$ no se puede dibujar, pero con $n = 3$ sí. Tomamos tres estudiantes.

In [ ]:
idx3 = [0, 3, 4]
x3, y3 = x[idx3], y[idx3]
X3 = matriz_disenio(x3)
beta3 = np.linalg.solve(X3.T @ X3, X3.T @ y3)
y3_hat = X3 @ beta3
e3 = y3 - y3_hat

print(f"x = {x3},  y = {y3}")
print(f"ŷ = {y3_hat.round(3)},  e = {e3.round(3)}")
print(f"e · 1 = {e3 @ np.ones(3):+.1e}   e · x = {e3 @ x3:+.1e}   (ortogonal al plano)")

In [ ]:
def unitario(v):
    return v / np.linalg.norm(v)


media3 = np.full(3, y3.mean())

# Los vectores y, ŷ y ȳ·1 miden ≈ 6 y el residual apenas unas décimas: dibujados desde el
# origen, el triángulo que interesa sería invisible. Hacemos zoom alrededor de ȳ·1, que
# también está en el plano. Como las notas rondan 3.6, x es casi paralelo a 1; el mismo plano
# lo generan 1 y x - x̄·1, que son perpendiculares, y con esa base se dibuja.
u1 = unitario(np.ones(3))
u2 = unitario(x3 - x3.mean())
radio = 0.8 * np.linalg.norm(y3 - media3)
s, t = np.meshgrid(np.linspace(-radio, radio, 10), np.linspace(-radio, radio, 10))
plano = media3 + s[..., None] * u1 + t[..., None] * u2

fig = plt.figure(figsize=(8, 7.5))
eje = fig.add_subplot(projection="3d")
eje.plot_surface(plano[..., 0], plano[..., 1], plano[..., 2], alpha=0.15, color="tab:purple")


def flecha(desde, hasta, color, etiqueta):
    d = np.asarray(hasta) - np.asarray(desde)
    eje.quiver(*desde, *d, color=color, arrow_length_ratio=0.1, linewidth=2.5)
    medio = np.asarray(desde) + d / 2
    eje.text(*medio, f"  {etiqueta}", color=color, fontsize=12)


flecha(media3, y3, "tab:green", r"$\mathbf{y}-\bar y\mathbf{1}$")
flecha(media3, y3_hat, "tab:orange", r"$\hat{\mathbf{y}}-\bar y\mathbf{1}$")
flecha(y3_hat, y3, "tab:red", r"$\mathbf{e}$")
for punto, nombre, color in [(media3, r"$\bar y\,\mathbf{1}$", "tab:green"),
                             (y3_hat, r"$\hat{\mathbf{y}}$", "tab:purple"),
                             (y3, r"$\mathbf{y}$", "black")]:
    eje.scatter(*punto, color=color, s=40, depthshade=False)
    eje.text(*punto, f"  {nombre}", color=color, fontsize=12)

# Marca de ángulo recto en ŷ
k = 0.15 * min(np.linalg.norm(e3), np.linalg.norm(y3_hat - media3))
ua, ub = unitario(media3 - y3_hat), unitario(e3)
esquina = np.array([y3_hat + k * ua, y3_hat + k * (ua + ub), y3_hat + k * ub])
eje.plot(*esquina.T, color="black", linewidth=1)

for fijar, centro in zip([eje.set_xlim, eje.set_ylim, eje.set_zlim], media3):
    fijar(centro - radio, centro + radio)
eje.set_box_aspect((1, 1, 1))  # misma escala en los tres ejes: el ángulo recto se ve recto
eje.set_xlabel("estudiante 1")
eje.set_ylabel("estudiante 2")
eje.set_zlabel("estudiante 3")
# Mirar casi de frente al triángulo (desde su normal), inclinado para que el plano no se vea de canto
normal = unitario(np.cross(y3 - media3, y3_hat - media3))
normal = normal if normal[2] >= 0 else -normal
eje.view_init(elev=np.degrees(np.arcsin(normal[2])) + 20,
              azim=np.degrees(np.arctan2(normal[1], normal[0])))
eje.set_title("Mínimos cuadrados como proyección ortogonal en $\\mathbb{R}^3$\n"
              "(plano morado: todas las predicciones $\\beta_0\\mathbf{1} + \\beta_1\\mathbf{x}$)")
plt.tight_layout()
plt.show()

El residual $\mathbf{e}$ cae **perpendicular** al plano: eso es exactamente
$\mathbf{X}^{\top}\mathbf{e} = \mathbf{0}$ de la sección 6. Y ahora la descomposición de la
sección 4 deja de ser un truco algebraico. Los tres vectores

- $\mathbf{y} - \bar y\mathbf{1}$ (de la media a los datos, el $\text{SS}_{\text{tot}}$),
- $\hat{\mathbf{y}} - \bar y\mathbf{1}$ (de la media a la predicción, en naranja, el
  $\text{SS}_{\text{reg}}$),
- $\mathbf{e} = \mathbf{y} - \hat{\mathbf{y}}$ (en rojo, el $\text{SS}_{\text{res}}$)

forman un **triángulo rectángulo**, porque el naranja está dentro del plano y el rojo es
perpendicular a él. $\text{SS}_{\text{tot}} = \text{SS}_{\text{reg}} +
\text{SS}_{\text{res}}$ es el **teorema de Pitágoras**, y

$$
R^2 = \frac{\lVert \hat{\mathbf{y}} - \bar y\mathbf{1} \rVert^2}{\lVert \mathbf{y} - \bar
y\mathbf{1} \rVert^2} = \cos^2\theta
$$

donde $\theta$ es el ángulo entre la hipotenusa y el cateto naranja. Un $R^2$ alto es un
$\mathbf{y}$ que casi cae dentro del plano. Con la recta a ojo $\hat{\mathbf{y}}$ no es la
proyección, el ángulo con $\mathbf{e}$ ya no es recto y Pitágoras no aplica — por eso las
áreas no cuadraban en la sección 4.

In [ ]:
hipotenusa = y3 - media3
cateto_reg = y3_hat - media3
cos_theta = hipotenusa @ cateto_reg / (np.linalg.norm(hipotenusa) * np.linalg.norm(cateto_reg))
r2_3 = 1 - np.sum(e3**2) / np.sum(hipotenusa**2)

print(f"‖y - ȳ1‖²              = {hipotenusa @ hipotenusa:.4f}")
print(f"‖ŷ - ȳ1‖² + ‖e‖²       = {cateto_reg @ cateto_reg + e3 @ e3:.4f}")
print(f"cateto_reg · e         = {cateto_reg @ e3:+.1e}  (ángulo recto)")
print(f"R² = {r2_3:.4f}   cos²θ = {cos_theta**2:.4f}   θ = {np.degrees(np.arccos(cos_theta)):.1f}°")

Esta vista escala sin cambios a cualquier número de predictores: con $p$ variables el
plano se vuelve un subespacio de dimensión $p+1$ dentro de $\mathbb{R}^n$, pero la recta
de mínimos cuadrados sigue siendo una proyección, el residual sigue siendo perpendicular y
el $R^2$ sigue siendo un coseno al cuadrado. También explica por qué el $R^2$ **nunca baja**
al agregar variables, como advierte `01-regresion-lineal.md`: un subespacio más grande solo
puede quedar igual de cerca o más cerca de $\mathbf{y}$.

## Resumen

| Lo que se vio | Su forma geométrica | Conecta con |
|---|---|---|
| Residual $e_i$ | Distancia **vertical** del punto a la recta | Análisis de residuales, `01-regresion-lineal.md` |
| SSE | Área total de los cuadrados de lado $\lvert e_i \rvert$ | Sensibilidad a outliers; MAE vs. RMSE (notebook 02) |
| $R^2$ | Fracción del área alrededor de $\bar y$ que la recta explica | Métricas, `01-regresion-lineal.md`; $R^2$ negativo en test |
| $\mathcal{L}(\beta_0, \beta_1)$ | Paraboloide; cada recta es un punto, OLS es el fondo | Descenso del gradiente (notebook 01) |
| Valle inclinado y alargado | Acoplamiento $\beta_0$–$\beta_1$; número de condición de $\mathbf{X}^{\top}\mathbf{X}$ | Estandarizar antes de descender (notebook 01); colinealidad (notebook 03) |
| Gradiente nulo | $\mathbf{X}^{\top}\mathbf{e} = \mathbf{0}$: la ecuación normal | `01-regresion-lineal.md`, sección 2 |
| 🔵 Proyección en $\mathbb{R}^n$ | $\text{SS}_{\text{tot}} = \text{SS}_{\text{reg}} + \text{SS}_{\text{res}}$ es Pitágoras; $R^2 = \cos^2\theta$ | $R^2$ ajustado; regularización (notebook 03) |